# Operator Inference

Operator inference aims to form a surrogate model for a dynamical system using only a set of trajectories from a possibly parameterized numerical simulation. It forms a surrogate model by finding the coefficients of a polynomial expansion of the trajectories, which involves solving a large least squares problem. To limit the size of the least squares problem these trajectories are often projected into a lower dimensional space before fitting the polynomial model. This space is typically found using the SVD. In this example, we will use `RandLinearAlgebra` to replace the SVD step with the RandomizedSVD. First, we will load the required packages and background code for the numerical simulations and operator inference.

In [1]:
using LinearAlgebra 
using RandLinearAlgebra
include("helpers/heat.jl")
include("helpers/opInf.jl")

GeneralBackwardEuler

Next we will perform the standard operator inference procedure and fit a degree 0, linear, polynomial to a set of compressed trajectories. We will consider our heat simulation over 10,000 time points at 4,095 spatial points on bar that is initially at a constant temperature and then has a heat of 1 applied at both ends throughout the simulation.

In [2]:
n_x = 2^12 - 1;
n_times = 10000;
delta_t = 1e-2;
mu = .1;
delta_x = 1 / (n_x + 1);
r = 11;
# use a linear model to estimate this
degree = 0;

# define initial conditions to be a vector of all zeros
ic = zeros(n_x);
# define the boundary conditions to be such that heat is appled on both sides
B = zeros(n_x);
scaling = mu / delta_x^2;
B[1] = scaling;
B[end] = scaling;

1. We will generate our trajectories by running Backwards Euler.

In [3]:
X_traj = simulate_1d_heat(ic, B, n_times, delta_t, delta_x, mu)

4095×10000 Matrix{Float64}:
 0.0  0.992309  0.99614   0.997105  …  1.0  1.0  1.0  1.0  1.0  1.0  1.0
 0.0  0.984678  0.99228   0.99421      1.0  1.0  1.0  1.0  1.0  1.0  1.0
 0.0  0.977105  0.98842   0.991315     1.0  1.0  1.0  1.0  1.0  1.0  1.0
 0.0  0.96959   0.984562  0.98842      1.0  1.0  1.0  1.0  1.0  1.0  1.0
 0.0  0.962134  0.980704  0.985525     1.0  1.0  1.0  1.0  1.0  1.0  1.0
 0.0  0.954734  0.976847  0.982631  …  1.0  1.0  1.0  1.0  1.0  1.0  1.0
 0.0  0.947392  0.972991  0.979737     1.0  1.0  1.0  1.0  1.0  1.0  1.0
 0.0  0.940106  0.969137  0.976844     1.0  1.0  1.0  1.0  1.0  1.0  1.0
 0.0  0.932876  0.965285  0.973951     1.0  1.0  1.0  1.0  1.0  1.0  1.0
 0.0  0.925701  0.961435  0.971058     1.0  1.0  1.0  1.0  1.0  1.0  1.0
 0.0  0.918582  0.957587  0.968166  …  1.0  1.0  1.0  1.0  1.0  1.0  1.0
 0.0  0.911517  0.953741  0.965275     1.0  1.0  1.0  1.0  1.0  1.0  1.0
 0.0  0.904507  0.949897  0.962384     1.0  1.0  1.0  1.0  1.0  1.0  1.0
 ⋮                     

2. We project the trajectories onto a lower dimensional space using the first 11 left singular vectors from the SVD.

In [4]:
svd_time = @elapsed U, S, _ = svd(X_traj);
Ur = U[:, 1:r];
Input_data = ones(size(X_traj, 2));

3. Generate and solve our least squares system using the projected trajectories.

In [5]:
op_time = @elapsed OP = op_inf_problem(Ur, X_traj, Input_data, degree, n_times, delta_t, delta_x, least_squares_solve);

4. Consider the performance of the model under the same heating conditions throughout the simulation, but where the bar no longer starts at constant temperature. Specifically, the end of the bar is much colder than the middle.

In [6]:
ic_new = -1 * ones(n_x);
ic_new[500:1000] .= .4;
proj_op_time = @elapsed Xint = GeneralBackwardEuler(OP[1][1:r, 1:r], OP[1][r + 1,:], Ur' * ic_new, delta_t, n_times);
true_op_time = @elapsed X_new = simulate_1d_heat(ic_new, B, n_times, delta_t, delta_x, mu);

Look at prediction error results and timings.

In [7]:
error = norm(X_new - (Ur * Xint)) / norm(X_new);

print("Computing the SVD took ")
printstyled(svd_time, color=:yellow, bold=true)
print(" seconds. \n")
print("Fitting the project operator took ", op_time," seconds.\n")
print("Simulating the true model took ", true_op_time, " seconds.\n")
print("Simulating the projected model took ", proj_op_time, " seconds.\n")
print("The subspace error is ", norm(X_traj - Ur * Ur' * X_traj), ".\n")
print("The relative state error ", error, ".\n")

Computing the SVD took 20.604452041 seconds. 
Fitting the project operator took 1.051618167 seconds.
Simulating the true model took 0.5390575 seconds.
Simulating the projected model took 0.468609584 seconds.
The subspace error is 4.901638968519129e-6.
The relative state error 0.009068337921787887.


Now we will see how the RandomizedSVD will accelerate this process. Here we will use the `rapproximate` function to compute the RandomizedSVD is step 2 then repeat steps 3-4. 

In [9]:

time_rand = @elapsed rand_svd = rapproximate(
    # defining the Randomized SVD with a SparseSign Compressor
    RandSVD(
        compressor = SparseSign(
            compression_dim = r + 4,
            cardinality = Right(),
            nnz = 2
        )
    ), 
    X_traj
);
# accessing the left singular vectors of the RandomizedSVD recipe
Ur_random = rand_svd.U[:, 1:r];
# Repeat steps 3 - 4 with Ur_random
Input_data = ones(size(X_traj, 2));

# 3. Find the operator on the projected system
OPr = op_inf_problem(Ur_random, X_traj, Input_data, degree, n_times, delta_t, delta_x, least_squares_solve);

# 4. choose new set of initial conditions and compare the projected to actual model
ic_new = -1 * ones(n_x);
ic_new[500:1000] .= .4;
Xint_random = GeneralBackwardEuler(OPr[1][1:r, 1:r], OPr[1][r + 1,:], Ur_random' * ic_new, delta_t, n_times);

Now we will display the same error and timing results.

In [10]:
error_rand = norm(X_new - (Ur_random * Xint_random)) / norm(X_new);
print("\n")
print("Computing the Randomized SVD took took ")
printstyled(time_rand, color=:yellow, bold=true)
print(" seconds. \n")
print("The random subspace error is ", norm(X_traj - Ur_random * Ur_random' * X_traj), ".\n")
print("The relative state error with the randomized svd is ", error_rand , ".\n")
print("\n")
printstyled("The svd speedup is ", (svd_time) / (time_rand), ".\n", color=:green, bold = true)
printstyled("The overall speedup is ", (svd_time + op_time + proj_op_time) / (time_rand + op_time + proj_op_time), ".\n",color=:blue, bold = true)


Computing the Randomized SVD took took 0.05505675 seconds. 
The random subspace error is 4.901640151974446e-6.
The relative state error with the randomized svd is 0.009068338122056912.

The svd speedup is 374.2402528481975.
The overall speedup is 14.044878736479106.
